In [ ]:
!pip install langgraph langchain langchain-groq pypdf plantuml

In [ ]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "YOUR_API_KEY_HERE"

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.2
)

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

file_name

In [ ]:
from pypdf import PdfReader

def load_requirements(file_path):
    if file_path.endswith(".txt"):
        with open(file_path, "r") as f:
            return f.read()

    elif file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        text = []
        for page in reader.pages:
            content = page.extract_text()
            if content:
                text.append(content)
        return "\n".join(text)

    else:
        raise ValueError("Unsupported file type")

requirements_text = load_requirements(file_name)

print(requirements_text[:1000])

In [ ]:
def extract_functionalities(text):
    prompt = f"""
    Extract DISTINCT system functionalities from the requirements.

    Rules:
    - Each functionality should be atomic
    - Avoid duplicates
    - Return as a numbered list

    Requirements:
    {text}
    """
    return llm.invoke(prompt).content

In [ ]:
def extract_interactions(functionality):
    prompt = f"""
    Identify interaction flow for this functionality.

    Provide:
    - Actors
    - System components
    - Step-by-step sequence

    Functionality:
    {functionality}
    """
    return llm.invoke(prompt).content

In [ ]:
def generate_sequence_diagram(interaction):
    prompt = f"""
    Convert this into a SysML sequence diagram using PlantUML.

    Rules:
    - Use @startuml / @enduml
    - Include actors and participants
    - Show message flow clearly
    - Keep it clean and valid

    Interaction:
    {interaction}
    """
    return llm.invoke(prompt).content

In [ ]:
def validate_diagram(diagram):
    prompt = f"""
    Validate and fix this PlantUML sequence diagram.

    Ensure:
    - Correct syntax
    - Logical flow
    - No missing participants

    Diagram:
    {diagram}
    """
    return llm.invoke(prompt).content

In [ ]:
from langgraph.graph import StateGraph

class State(dict):
    pass


def functionality_node(state):
    state["functionalities"] = extract_functionalities(state["requirements"])
    return state


def interaction_node(state):
    funcs = state["functionalities"].split("\n")
    interactions = []

    for f in funcs:
        if f.strip():
            interactions.append(extract_interactions(f))

    state["interactions"] = interactions
    return state


def diagram_node(state):
    diagrams = []

    for interaction in state["interactions"]:
        raw = generate_sequence_diagram(interaction)
        fixed = validate_diagram(raw)
        diagrams.append(fixed)

    state["diagrams"] = diagrams
    return state


builder = StateGraph(State)

builder.add_node("functionality", functionality_node)
builder.add_node("interaction", interaction_node)
builder.add_node("diagram", diagram_node)

builder.set_entry_point("functionality")

builder.add_edge("functionality", "interaction")
builder.add_edge("interaction", "diagram")

graph = builder.compile()

In [ ]:
state = {
    "requirements": requirements_text
}

result = graph.invoke(state)

diagrams = result["diagrams"]

print(f"Generated {len(diagrams)} diagrams")

In [ ]:
for i, d in enumerate(diagrams):
    print(f"\n===== Diagram {i+1} =====\n")
    print(d)

In [ ]:
from plantuml import PlantUML

plantuml = PlantUML(url="http://www.plantuml.com/plantuml/img/")

for i, diagram in enumerate(diagrams):
    file_name = f"diagram_{i}.txt"

    with open(file_name, "w") as f:
        f.write(diagram)

    plantuml.processes_file(file_name)